# Experiment: K-Fold Results Review

Objective:
- Load the CSV outputs from a completed `run_resnet50_kfold_decoding(...)` run.
- Inspect layer-wise performance, condition-wise behavior, optional vertical-image boundary responses, and runtime.
- Keep the review lightweight so it is easy to reuse after each new run.


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("default")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)


## Configure the saved run

Point `output_dir` at the folder where your real run wrote CSV outputs.

Set `prefix` to match the saved file prefix. Common examples:
- `resnet50_all_layers_kfold`
- `resnet50_layer4_last_kfold`


In [ ]:
output_dir = Path("results_kfold")
prefix = "resnet50_all_layers_kfold"  # update after a real run if needed


def load_required_csv(name: str) -> pd.DataFrame:
    path = output_dir / f"{prefix}_{name}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    return pd.read_csv(path)


def load_optional_csv(name: str) -> pd.DataFrame | None:
    path = output_dir / f"{prefix}_{name}.csv"
    if not path.exists():
        return None
    return pd.read_csv(path)


trial_df = load_required_csv("trial_outputs")
fold_metrics_df = load_required_csv("fold_metrics")
layer_summary_df = load_required_csv("layer_summary")
boundary_vertical_df = load_optional_csv("vertical_boundary_outputs")
boundary_summary_df = load_optional_csv("vertical_boundary_summary")
timing_df = load_optional_csv("timing")

loaded_shapes = {
    "trial_df": trial_df.shape,
    "fold_metrics_df": fold_metrics_df.shape,
    "layer_summary_df": layer_summary_df.shape,
    "boundary_vertical_df": None if boundary_vertical_df is None else boundary_vertical_df.shape,
    "boundary_summary_df": None if boundary_summary_df is None else boundary_summary_df.shape,
    "timing_df": None if timing_df is None else timing_df.shape,
}
loaded_shapes


## Layer-wise summary

Start with the aggregated view across folds. This is the quickest way to see which checkpoint is strongest.


In [ ]:
layer_summary_df = layer_summary_df.sort_values("balanced_accuracy_mean", ascending=False).reset_index(drop=True)
layer_summary_df


In [ ]:
ordered = layer_summary_df.copy()
x = np.arange(len(ordered))

fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(
    x,
    ordered["balanced_accuracy_mean"],
    yerr=ordered["balanced_accuracy_std"],
    fmt="o-",
    capsize=4,
    linewidth=2,
)
ax.set_xticks(x)
ax.set_xticklabels(ordered["layer_name"], rotation=30, ha="right")
ax.set_ylabel("Balanced Accuracy")
ax.set_title("Layer-wise performance across folds")
ax.set_ylim(0.0, 1.05)
ax.grid(alpha=0.3)
plt.show()


## Condition-wise behavior on held-out test images

Use the trial-level table to break performance down by stimulus properties. The example below computes accuracy and CW choice rate by condition.


In [ ]:
condition_df = (
    trial_df.groupby(["layer_name", "mean", "sd", "ss"], dropna=False)
    .agg(
        n=("correct", "size"),
        accuracy=("correct", "mean"),
        cw_rate=("pred_class", "mean"),
        decision_value_mean=("decision_value", "mean"),
    )
    .reset_index()
)
condition_df.head(12)


In [ ]:
selected_layer = layer_summary_df.iloc[0]["layer_name"]
plot_df = condition_df.loc[condition_df["layer_name"] == selected_layer].copy()
plot_df = plot_df.sort_values(["ss", "sd", "mean"]).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

for (sd, ss), group in plot_df.groupby(["sd", "ss"]):
    label = f"sd={sd}, ss={ss}"
    axes[0].plot(group["mean"], group["accuracy"], marker="o", label=label)
    axes[1].plot(group["mean"], group["cw_rate"], marker="o", label=label)

axes[0].set_title(f"Accuracy by mean ({selected_layer})")
axes[0].set_xlabel("Mean orientation")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0.0, 1.05)
axes[0].grid(alpha=0.3)

axes[1].set_title(f"CW choice rate by mean ({selected_layer})")
axes[1].set_xlabel("Mean orientation")
axes[1].set_ylabel("CW choice rate")
axes[1].set_ylim(0.0, 1.05)
axes[1].grid(alpha=0.3)
axes[1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5))

plt.show()


## Optional vertical-image boundary review

If you enabled `evaluate_vertical=True`, these cells summarize how each fold's trained decoder treated `mean=0` images.


In [ ]:
if boundary_summary_df is None or boundary_summary_df.empty:
    boundary_preview_df = None
    print("No vertical boundary summary was found for this run.")
else:
    boundary_preview_df = boundary_summary_df.sort_values(["layer_name", "fold_id"]).reset_index(drop=True)

boundary_preview_df


In [ ]:
if boundary_summary_df is None or boundary_summary_df.empty:
    print("Skip: no boundary output file is present.")
else:
    boundary_plot_df = (
        boundary_summary_df.groupby("layer_name", dropna=False)
        .agg(
            cw_rate_mean=("cw_rate", "mean"),
            cw_rate_std=("cw_rate", "std"),
            decision_value_mean=("decision_value_mean", "mean"),
        )
        .reset_index()
    )
    boundary_plot_df["cw_rate_std"] = boundary_plot_df["cw_rate_std"].fillna(0.0)

    x = np.arange(len(boundary_plot_df))
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)

    axes[0].bar(x, boundary_plot_df["cw_rate_mean"], yerr=boundary_plot_df["cw_rate_std"], capsize=4)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(boundary_plot_df["layer_name"], rotation=30, ha="right")
    axes[0].set_ylabel("CW rate on mean=0")
    axes[0].set_ylim(0.0, 1.05)
    axes[0].grid(alpha=0.3)

    axes[1].bar(x, boundary_plot_df["decision_value_mean"])
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(boundary_plot_df["layer_name"], rotation=30, ha="right")
    axes[1].set_ylabel("Mean decision value")
    axes[1].grid(alpha=0.3)

    plt.show()


## Runtime review

The timing table helps estimate which parts of the pipeline dominate runtime when you scale up dataset size or switch layers.


In [ ]:
if timing_df is None or timing_df.empty:
    timing_preview_df = None
    print("No timing CSV was found for this run.")
else:
    timing_preview_df = timing_df.copy()

timing_preview_df


In [ ]:
if timing_df is None or timing_df.empty:
    print("Skip: no timing output file is present.")
else:
    layer_timing_df = timing_df.loc[timing_df["stage"] == "layer_decode"].copy()
    if layer_timing_df.empty:
        print("No per-layer timing rows were recorded.")
    else:
        layer_timing_df = layer_timing_df.sort_values("seconds", ascending=False).reset_index(drop=True)
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(layer_timing_df["layer_name"], layer_timing_df["seconds"])
        ax.set_ylabel("Seconds")
        ax.set_title("Per-layer runtime")
        ax.grid(axis="y", alpha=0.3)
        plt.xticks(rotation=30, ha="right")
        plt.show()

    total_seconds = timing_df.loc[timing_df["stage"] == "total", "seconds"]
    if not total_seconds.empty:
        print(f"Total runtime: {total_seconds.iloc[0]:.3f} s")


## Next steps

- Change `prefix` to compare different saved runs.
- Swap `selected_layer` in the plotting cells if you want to inspect a non-best layer.
- Add custom grouping variables if you want to split results by only `sd`, only `ss`, or by instance subsets.
